# 4k Video Upscaler (Real-ESRGAN)

Upscale your videos up to 4k on free Google Colab or locally using [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN).

**Features:**
- Support for **Google Drive** and **Local Upload**.
- Multiple models: `RealESRGAN_x4plus`, `RealESRGAN_x4plus_anime_6B`, `realesr-animevideov3`.
- Flexible resolution: `FHD`, `2K`, `4K`, or custom multipliers.
- Works in **Google Colab** and **Local Python** environment.

**Github:** [pareshmishra23/4k-genration](https://github.com/pareshmishra23/4k-genration)

---
**Note:** In Colab, make sure to change Runtime to GPU - `Runtime` -> `Change runtime type` -> `T4 GPU`.


# 1. Setup (~1 minute)


In [ ]:
import os, sys, subprocess, pathlib
import torch

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    if not torch.cuda.is_available():
        print("WARNING: GPU not detected. Using CPU (will be very slow).")
    else:
        print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("Running in Local Environment")

# Clone repository if not already present
if not os.path.exists('4k-genration'):
    !git clone https://github.com/pareshmishra23/4k-genration.git
    %cd 4k-genration

# Install dependencies from requirements.txt
!pip install -q -r requirements.txt

# Clone Real-ESRGAN if not present and install it
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git
    %cd Real-ESRGAN
    !pip install -q .
    %cd ..

# Ensure we are in the correct directory for upscale_video.py
if not os.path.exists('upscale_video.py') and os.path.exists('4k-genration/upscale_video.py'):
    %cd 4k-genration

print("Setup complete!")


# 2. Input Selection

Choose how you want to provide the video file.


In [ ]:
input_method = "Upload" #@param ["Upload", "Google Drive", "Local Path"]
video_path = "" 

if input_method == "Google Drive" and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    video_path = "/content/gdrive/MyDrive/video.mp4" #@param {type:"string"}
    print("Drive mounted. Please ensure the path is correct.")
elif input_method == "Upload" and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        video_path = os.path.abspath(fn)
        print(f'User uploaded file {fn} with length {len(uploaded[fn])} bytes')
elif input_method == "Local Path":
    video_path = "/path/to/your/video.mp4" #@param {type:"string"}
    print("Using local path. Ensure it's accessible.")
else:
    if not IN_COLAB:
        print("Upload/Drive options are for Colab. Please use 'Local Path'.")
        video_path = "/path/to/your/video.mp4" # Default for local if not Colab
    else:
        print("Please select an input method.")

if video_path and not os.path.exists(video_path):
    print(f"Error: Video file not found at {video_path}. Please check the path or upload again.")
else:
    print(f"Input video set to: {video_path}")


# 3. Upscale Configuration

Configure your settings and run the upscaler.


In [ ]:
#@title Configuration { display-mode: "form" }

output_directory = "./output" #@param {type:"string"}
resolution = "4k (3840 x 2160)" #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "2 x original", "3 x original", "4 x original"]
model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "RealESRNet_x4plus", "RealESRGAN_x2plus", "realesr-general-x4v3"]
tile_size = 0 #@param {type:"integer"}

if video_path and os.path.exists(video_path):
    print(f"Starting upscaling for: {video_path}")
    !python upscale_video.py --input "{video_path}" --output "{output_directory}" --resolution "{resolution}" --model "{model}" --tile {tile_size}
else:
    print("Cannot start upscaling: Input video path is not valid or not set.")


# 4. Download Results

If you are in Colab, you can download the upscaled video here.


In [ ]:
if IN_COLAB:
    import glob
    from google.colab import files
    output_files = glob.glob(os.path.join(output_directory, "*_upscaled_*.mp4"))
    if output_files:
        latest_file = max(output_files, key=os.path.getctime)
        print(f"Downloading {latest_file}...")
        files.download(latest_file)
    else:
        print("No upscaled videos found in output directory.")
else:
    print(f"Results are in {os.path.abspath(output_directory)}. ")
